# Tiny Aya → Wazuh alert analyst (AlignTune LoRA)
Runtime: GPU (T4 is enough with 4-bit). Before running: accept the licence on https://huggingface.co/CohereLabs/tiny-aya-global and have a Hugging Face **write** token ready.

Set the two variables in the next cell.

In [ ]:
HF_USER = "Yusheng88"
GITHUB_REPO = "https://github.com/Patupi1/wazuh-alert-analyst"
DATASET = f"{HF_USER}/wazuh-alert-analyst"
ADAPTER_REPO = f"{HF_USER}/tiny-aya-wazuh-analyst-lora"
GGUF_REPO = f"{HF_USER}/tiny-aya-wazuh-analyst-GGUF"

In [ ]:
%pip -q install aligntune auditkit
from huggingface_hub import login; login()
!git clone $GITHUB_REPO repo 2>/dev/null || (cd repo && git pull)
!ls repo

In [ ]:
# QUICK CHECK first: MAX_STEPS = -1. Then MAX_STEPS = -1 for the full run (1 epoch = ~137 steps; ~75 s/step on a T4 => ~3 h).
MAX_STEPS = -1
!cd repo && python train/train_sft.py --dataset $DATASET --out /content/out --epochs 1 --max-steps $MAX_STEPS

In [ ]:
# 1) save the adapter first (the runtime can die during export)
from huggingface_hub import HfApi; api = HfApi()
api.create_repo(ADAPTER_REPO, exist_ok=True)
api.upload_folder(folder_path='/content/out', repo_id=ADAPTER_REPO,
                  allow_patterns=['adapter_*', '*.json', 'tokenizer*', '*.jinja', 'README.md'], ignore_patterns=['checkpoint-*/**'])
# 2) merge with AlignTune (it writes to <output>/merged)
!aligntune export merge-adapter /content/out --output /content/out/merged
# 3) GGUF with llama.cpp directly (AlignTune's gguf exporter 404s on a renamed converter script)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install /content/llama.cpp/gguf-py
!mkdir -p /content/out/gguf
!python /content/llama.cpp/convert_hf_to_gguf.py /content/out/merged/merged --outtype f16 --outfile /content/out/gguf/tiny-aya-wazuh-f16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/out/gguf/tiny-aya-wazuh-f16.gguf /content/out/gguf/tiny-aya-wazuh-analyst-Q4_K_M.gguf Q4_K_M
!rm /content/out/gguf/tiny-aya-wazuh-f16.gguf
# 4) upload the GGUF
api.create_repo(GGUF_REPO, exist_ok=True)
api.upload_file(path_or_fileobj='/content/out/gguf/tiny-aya-wazuh-analyst-Q4_K_M.gguf', path_in_repo='tiny-aya-wazuh-analyst-Q4_K_M.gguf', repo_id=GGUF_REPO)

Then on the homelab box: `ollama pull hf.co/<HF_USER>/tiny-aya-wazuh-analyst-GGUF:Q4_K_M` and run `src/eval.py` with `--model ollama:hf.co/<HF_USER>/tiny-aya-wazuh-analyst-GGUF:Q4_K_M` on the same test and attack sets as the baseline.